## Interact with the MultiversX Network using the Python SDK

### Query account state

#### Get EGLD balance

In [ ]:
from multiversx_sdk import Address, DevnetEntrypoint

entrypoint = DevnetEntrypoint()
api = entrypoint.create_network_provider()

alice = Address.new_from_bech32("erd1qyu5wthldzr8wx5c9ucg8kjagg0jfs53s8nr3zpz3hypefsdd8ssycr6th")
account = api.get_account(address=alice)

print("EGLD Balance: ", account.balance)
print("Nonce: ", account.nonce)

#### Get ESDT token balance

In [ ]:
from multiversx_sdk import Address, DevnetEntrypoint, Token

entrypoint = DevnetEntrypoint()
api = entrypoint.create_network_provider()

alice = Address.new_from_bech32("erd1qyu5wthldzr8wx5c9ucg8kjagg0jfs53s8nr3zpz3hypefsdd8ssycr6th")
token = Token(identifier="CHOCOLATE-e98f2b")

token_amount = api.get_token_of_account(address=alice, token=token)

print(f"ESDT Token Balance for {token.identifier}: ", token_amount.amount)

### Token Transfers

#### EGLD Transfers using the Controller

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, Address, DevnetEntrypoint

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

grace = Address.new_from_bech32("erd1r69gk66fmedhhcg24g2c5kn2f2a5k4kvpr6jfw67dn2lyydd8cfswy6ede")

transfers_controller = entrypoint.create_transfers_controller()
transaction = transfers_controller.create_transaction_for_transfer(
    sender=alice,
    nonce=alice.nonce,
    receiver=grace,
    native_transfer_amount=1_000_000_000_000,
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

#### EGLD Transfers using the Factory

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, Address, DevnetEntrypoint

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

grace = Address.new_from_bech32("erd1r69gk66fmedhhcg24g2c5kn2f2a5k4kvpr6jfw67dn2lyydd8cfswy6ede")

transfers_factory = entrypoint.create_transfers_transactions_factory()
transaction = transfers_factory.create_transaction_for_transfer(
    sender=alice.address,
    receiver=grace,
    native_amount=1_000_000_000_000,
)

transaction.nonce = alice.nonce
transaction.signature = alice.sign_transaction(transaction)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

#### ESDT Token Transfers using the Controller

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, Address, DevnetEntrypoint, Token, TokenTransfer

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

grace = Address.new_from_bech32("erd1r69gk66fmedhhcg24g2c5kn2f2a5k4kvpr6jfw67dn2lyydd8cfswy6ede")
token = Token(identifier="CHOCOLATE-e98f2b")
token_transfers = TokenTransfer(token=token, amount=1000)  # 10 tokens with 2 decimals

transfers_controller = entrypoint.create_transfers_controller()
transaction = transfers_controller.create_transaction_for_transfer(
    sender=alice,
    nonce=alice.nonce,
    receiver=grace,
    token_transfers=[token_transfers]
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

#### ESDT Token Transfers using the Factory

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, Address, DevnetEntrypoint, Token, TokenTransfer

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

grace = Address.new_from_bech32("erd1r69gk66fmedhhcg24g2c5kn2f2a5k4kvpr6jfw67dn2lyydd8cfswy6ede")
token = Token(identifier="CHOCOLATE-e98f2b")
token_transfers = TokenTransfer(token=token, amount=1000)  # 10 tokens with 2 decimals

transfers_factory = entrypoint.create_transfers_transactions_factory()
transaction = transfers_factory.create_transaction_for_transfer(
    sender=alice.address,
    receiver=grace,
    token_transfers=[token_transfers]
)

transaction.nonce = alice.nonce
transaction.signature = alice.sign_transaction(transaction)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

### Smart Contracts

#### Interact with the Adder contract without ABI File

##### Deploy

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint
from multiversx_sdk.abi import BigUIntValue

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

adder_bytecode = Path("contracts/adder.wasm")

sc_controller = entrypoint.create_smart_contract_controller()
transaction = sc_controller.create_transaction_for_deploy(
    sender=alice,
    nonce=alice.nonce,
    bytecode=adder_bytecode,
    arguments=[BigUIntValue(0)]
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

contract = sc_controller.await_completed_deploy(tx_hash)
print("Contract address: ", contract.contracts[0].address)

In [ ]:
from multiversx_sdk import Address

ADDER_ADDRESS = Address.new_from_bech32("erd1qqqqqqqqqqqqqpgqqdd3vqv57tkzrfauzpdk89ury879j2a6d8ss6r82tc")

##### Call the "add" function

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint
from multiversx_sdk.abi import BigUIntValue

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

sc_controller = entrypoint.create_smart_contract_controller()
transaction = sc_controller.create_transaction_for_execute(
    sender=alice,
    nonce=alice.nonce,
    contract=ADDER_ADDRESS,
    function="add",
    arguments=[BigUIntValue(7)]
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

##### Query the contract (read contract state)

In [ ]:
from multiversx_sdk import DevnetEntrypoint

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

sc_controller = entrypoint.create_smart_contract_controller()
response = sc_controller.query(
    contract=ADDER_ADDRESS,
    function="getSum",
    arguments=[],
)

print(response)

#### Interact with the Lottery contract using the ABI file

##### Deploy the contract

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

bytecode = Path("contracts/lottery-esdt.wasm")
abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))

sc_controller = entrypoint.create_smart_contract_controller(abi=abi)
transaction = sc_controller.create_transaction_for_deploy(
    sender=alice,
    nonce=alice.nonce,
    bytecode=bytecode,
    arguments=[]
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

contract = sc_controller.await_completed_deploy(tx_hash)
print("Contract address: ", contract.contracts[0].address)

In [ ]:
from multiversx_sdk import Address

LOTTERY_ADDRESS = Address.new_from_bech32("erd1qqqqqqqqqqqqqpgqg2fxwet8jkj8ss3u9wukv86qvuul8smmd8ss4h3x64")

#### Start the lottery

In [ ]:
import time
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))

timestamp = int(time.time())

sc_controller = entrypoint.create_smart_contract_controller(abi=abi)
transaction = sc_controller.create_transaction_for_execute(
    sender=alice,
    nonce=alice.nonce,
    contract=LOTTERY_ADDRESS,
    function="start",
    arguments=["testLottery", "ALICE-5627f1", 1, None, timestamp + 120, None, None, None, None]
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

##### Get lottery info

In [ ]:
from pathlib import Path
from multiversx_sdk import DevnetEntrypoint
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))

sc_controller = entrypoint.create_smart_contract_controller(abi=abi)
response = sc_controller.query(
    contract=LOTTERY_ADDRESS,
    function="getLotteryInfo",
    arguments=["testLottery"]
)

print(response)

##### Buy tickets using Alice

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint, Token, TokenTransfer
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True, gas_limit_multiplier=1.2)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))
sc_controller = entrypoint.create_smart_contract_controller(abi=abi)

token = Token(identifier="ALICE-5627f1")
transfer = TokenTransfer(token=token, amount=1)

transaction = sc_controller.create_transaction_for_execute(
    sender=alice,
    nonce=alice.nonce,
    contract=LOTTERY_ADDRESS,
    function="buy_ticket",
    arguments=["testLottery"],
    token_transfers=[transfer],
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

##### Buy tickets using Grace

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint, Token, TokenTransfer
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True, gas_limit_multiplier=1.2)

grace_pem=Path("~/multiversx-sdk/testwallets/latest/users/grace.pem").expanduser()
grace = Account.new_from_pem(grace_pem)
grace.nonce = entrypoint.recall_account_nonce(grace.address)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))
sc_factory = entrypoint.create_smart_contract_transactions_factory(abi=abi)

token = Token(identifier="ALICE-5627f1")
transfer = TokenTransfer(token=token, amount=1)

transaction = sc_factory.create_transaction_for_execute(
    sender=grace.address,
    contract=LOTTERY_ADDRESS,
    function="buy_ticket",
    arguments=["testLottery"],
    token_transfers=[transfer],
)

transaction.nonce = grace.nonce
transaction.signature = grace.sign_transaction(transaction)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")

##### Check lottery status

In [ ]:
from pathlib import Path
from multiversx_sdk import DevnetEntrypoint
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))

sc_controller = entrypoint.create_smart_contract_controller(abi=abi)
response = sc_controller.query(
    contract=LOTTERY_ADDRESS,
    function="status",
    arguments=["testLottery"]
)

print(response)

##### Deterimine lottery winner

In [ ]:
from pathlib import Path
from multiversx_sdk import Account, DevnetEntrypoint
from multiversx_sdk.abi import Abi

entrypoint = DevnetEntrypoint(with_gas_limit_estimator=True, gas_limit_multiplier=1.2)

alice_pem=Path("~/multiversx-sdk/testwallets/latest/users/alice.pem").expanduser()
alice = Account.new_from_pem(alice_pem)
alice.nonce = entrypoint.recall_account_nonce(alice.address)

abi = Abi.load(Path("contracts/lottery-esdt.abi.json"))
sc_controller = entrypoint.create_smart_contract_controller(abi=abi)

transaction = sc_controller.create_transaction_for_execute(
    sender=alice,
    nonce=alice.nonce,
    contract=LOTTERY_ADDRESS,
    function="determine_winner",
    arguments=["testLottery"],
)

tx_hash = entrypoint.send_transaction(transaction)
print(f"Transaction sent with hash: {tx_hash.hex()}")